# EMT three-phase VSI GFM staged load-step simulation

This notebook is the Python/pybind equivalent of the C++ example
`EMT_Ph3_GFM_Test.cpp`.

The setup contains:

- a single-phase power-flow initialization,
- a three-phase EMT voltage-source inverter with VCO-based voltage control,
- two switched RX loads,
- switching events at 0.2 s, 0.4 s, 0.6 s, and 0.8 s,
- CSV logging and result plots.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

import dpsimpy

emt = dpsimpy.emt
sp = dpsimpy.sp

ph3 = emt.ph3
sp_ph1 = sp.ph1

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
PhaseType = dpsimpy.PhaseType
PowerflowBusType = dpsimpy.PowerflowBusType
Solver = dpsimpy.Solver
SolverBehaviour = dpsimpy.SolverBehaviour
SwitchEvent3Ph = dpsimpy.event.SwitchEvent3Ph

## Simulation parameters

In [ ]:
final_time = 1.0
time_step = 100e-6

sim_name = "EMT_VSI_GFM_LoadStep"
sim_name_pf = sim_name + "_PF"
sim_name_emt = sim_name + "_EMT"

pv_with_control = True

## Yazdani test-system parameters

The C++ example obtains these values from
`CIM::Examples::Components::GFM::Derived Yazdani`. They are written explicitly
here so that the notebook is self-contained.


In [ ]:
system_frequency = 60.0
omega_nominal = 2.0 * np.pi * system_frequency

line_to_line_voltage_rms = 400.0
phase_voltage_rms = line_to_line_voltage_rms / np.sqrt(3.0)
phase_voltage_peak = np.sqrt(2.0) * phase_voltage_rms

# Filter parameters
lf = 100e-6
cf = 2.5e-3
rf = 2.07e-3
rc = 0.1

# Voltage and current controller gains
kp_voltage_ctrl = 0.5
ki_voltage_ctrl = 200.0
kp_current_ctrl = 5.0
ki_current_ctrl = 2000.0

# dq voltage references
vd_ref = 400.0
vq_ref = 0.0

# Initial controller states
phi_d_init = 0.0
phi_q_init = 0.0
gamma_d_init = 0.0
gamma_q_init = 0.0

# Load impedances from the corresponding Yazdani test setup
res1 = 83e-3
ind1 = 137e-6

res2 = 83e-3
ind2 = 137e-6
cap2 = 1e-3

connection_resistance = 0.88e-3

switch_open_resistance = 1e9
switch_closed_resistance = 1e-9

## Calculate the two load operating points

In [ ]:
omega = 2.0 * np.pi * system_frequency

load1_impedance = complex(
    res1,
    ind1 * omega,
)

load2_impedance = complex(
    res2,
    ind2 * omega - 1.0 / (omega * cap2),
)

# This reproduces the formula used by the C++ example.
load1_s = 3.0 * line_to_line_voltage_rms**2 / load1_impedance
load2_s = 3.0 * line_to_line_voltage_rms**2 / load2_impedance

load1_p = load1_s.real
load1_q = load1_s.imag

load2_p = load2_s.real
load2_q = load2_s.imag

print(f"Load 1: P = {load1_p:.6f} W, Q = {load1_q:.6f} var")
print(f"Load 2: P = {load2_p:.6f} W, Q = {load2_q:.6f} var")

## Helper functions

In [ ]:
def three_phase_parameter(value):
    return dpsimpy.Math.single_phase_parameter_to_three_phase(value)


def three_phase_power(value):
    return dpsimpy.Math.single_phase_power_to_three_phase(value)


def log_attr(logger, name, obj, attr_name):
    if hasattr(obj, "attr"):
        logger.log_attribute(name, obj.attr(attr_name))
    else:
        logger.log_attribute(name, obj.attribute(attr_name))


def load_csv(name):
    path = Path("logs") / name / f"{name}.csv"
    return pd.read_csv(path, skipinitialspace=True)


def signal_columns(df, prefix):
    return [
        column
        for column in df.columns
        if column == prefix or column.startswith(prefix + "_")
    ]

# Power-flow initialization

In [ ]:
def build_and_run_powerflow():
    Logger.set_log_dir("logs/" + sim_name_pf)

    time_step_pf = final_time
    final_time_pf = final_time + time_step_pf

    n1_pf = sp.SimNode("n1", PhaseType.Single)
    n2_pf = sp.SimNode("n2", PhaseType.Single)

    extnet_pf = sp_ph1.NetworkInjection("Slack")
    extnet_pf.set_parameters(line_to_line_voltage_rms)
    extnet_pf.set_base_voltage(line_to_line_voltage_rms)
    extnet_pf.modify_power_flow_bus_type(PowerflowBusType.VD)

    line_pf = sp_ph1.PiLine("PiLine")
    line_pf.set_parameters(
        connection_resistance,
        0.0,
        0.0,
    )
    line_pf.set_base_voltage(line_to_line_voltage_rms)

    extnet_pf.connect([n1_pf])
    line_pf.connect([n1_pf, n2_pf])

    system_pf = SystemTopology(
        system_frequency,
        [n1_pf, n2_pf],
        [line_pf, extnet_pf],
    )

    logger_pf = Logger(sim_name_pf)
    log_attr(logger_pf, "v1", n1_pf, "v")
    log_attr(logger_pf, "v2", n2_pf, "v")

    sim_pf = Simulation(sim_name_pf)
    sim_pf.set_system(system_pf)
    sim_pf.set_time_step(time_step_pf)
    sim_pf.set_final_time(final_time_pf)
    sim_pf.set_domain(Domain.SP)
    sim_pf.set_solver(Solver.NRP)
    sim_pf.set_solver_component_behaviour(SolverBehaviour.Initialization)
    sim_pf.do_init_from_nodes_and_terminals(False)
    sim_pf.add_logger(logger_pf)
    sim_pf.run()

    return system_pf, line_pf


system_pf, line_pf = build_and_run_powerflow()

# Dynamic EMT simulation

In [ ]:
def build_and_run_emt(system_pf, line_pf):
    Logger.set_log_dir("logs/" + sim_name_emt)

    n1_emt = emt.SimNode("n1", PhaseType.ABC)
    n2_emt = emt.SimNode("n2", PhaseType.ABC)
    n3_emt = emt.SimNode("n3", PhaseType.ABC)
    n4_emt = emt.SimNode("n4", PhaseType.ABC)

    load_emt_1 = ph3.RXLoad("Load1")
    load_emt_1.set_parameters(
        three_phase_power(load1_p),
        three_phase_power(load1_q),
        line_to_line_voltage_rms,
    )

    load_emt_2 = ph3.RXLoad("Load2")
    load_emt_2.set_parameters(
        three_phase_power(load2_p),
        three_phase_power(load2_q),
        line_to_line_voltage_rms,
    )

    connection_emt = ph3.Resistor("ResOn")
    connection_emt.set_parameters(three_phase_parameter(connection_resistance))

    pv = ph3.VSIVoltageControlVCO(
        "pv",
        "pv",
        dpsimpy.LoggerLevel.debug,
        False,
    )

    pv.set_parameters(
        omega_nominal,
        vd_ref,
        vq_ref,
    )

    pv.set_controller_parameters(
        kp_voltage_ctrl,
        ki_voltage_ctrl,
        kp_current_ctrl,
        ki_current_ctrl,
        omega_nominal,
    )

    pv.set_filter_parameters(
        lf,
        cf,
        rf,
        rc,
    )

    pv.set_initial_state_values(
        phi_d_init,
        phi_q_init,
        gamma_d_init,
        gamma_q_init,
    )

    pv.with_control(pv_with_control)

    switch_1 = ph3.Switch("Switch1")
    switch_1.set_parameters(
        three_phase_parameter(switch_open_resistance),
        three_phase_parameter(switch_closed_resistance),
    )
    switch_1.open()

    switch_2 = ph3.Switch("Switch2")
    switch_2.set_parameters(
        three_phase_parameter(switch_open_resistance),
        three_phase_parameter(switch_closed_resistance),
    )
    switch_2.open()

    pv.connect([n1_emt])
    connection_emt.connect([n1_emt, n2_emt])
    switch_1.connect([n2_emt, n3_emt])
    load_emt_1.connect([n3_emt])
    switch_2.connect([n3_emt, n4_emt])
    load_emt_2.connect([n4_emt])

    system_emt = SystemTopology(
        system_frequency,
        [n1_emt, n2_emt, n3_emt, n4_emt],
        [
            connection_emt,
            switch_1,
            switch_2,
            load_emt_1,
            load_emt_2,
            pv,
        ],
    )

    system_emt.init_with_powerflow(system_pf, Domain.EMT)

    # Equivalent of:
    # linePF->attributeTyped<Real>("p_inj")->get()
    # linePF->attributeTyped<Real>("q_inj")->get()
    initial_active_power = line_pf.attr("p_inj").get()
    initial_reactive_power = line_pf.attr("q_inj").get()

    # This requires terminal access to be exposed in the component binding.
    # Keep the following line if your pybind interface exposes terminal().
    try:
        pv.terminal(0).set_power(complex(initial_active_power, initial_reactive_power))
    except (AttributeError, TypeError):
        print(
            "Note: pv.terminal(0).set_power(...) is not exposed through pybind. "
            "The EMT model will use node/terminal initialization only."
        )

    logger_emt = Logger(sim_name_emt)

    log_attr(logger_emt, "Voltage_PCC", n1_emt, "v")
    log_attr(logger_emt, "Voltage_Node_2", n2_emt, "v")
    log_attr(logger_emt, "Voltage_Node_3", n3_emt, "v")
    log_attr(logger_emt, "Voltage_Node_4", n4_emt, "v")

    log_attr(logger_emt, "Voltage_Source", pv, "Vs")
    log_attr(logger_emt, "Current_RLC", pv, "i_intf")
    log_attr(logger_emt, "P_elec", pv, "P_elec")
    log_attr(logger_emt, "Q_elec", pv, "Q_elec")

    sim = Simulation(sim_name_emt)

    sim.add_event(SwitchEvent3Ph(0.2, switch_1, True))
    sim.add_event(SwitchEvent3Ph(0.8, switch_1, False))

    sim.add_event(SwitchEvent3Ph(0.4, switch_2, True))
    sim.add_event(SwitchEvent3Ph(0.6, switch_2, False))

    sim.set_system(system_emt)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time + time_step)
    sim.set_domain(Domain.EMT)
    sim.set_solver(Solver.MNA)
    sim.do_system_matrix_recomputation(True)
    sim.do_init_from_nodes_and_terminals(True)
    sim.add_logger(logger_emt)
    sim.run()

    return system_emt, pv


system_emt, pv = build_and_run_emt(system_pf, line_pf)

# Load simulation results

In [ ]:
pf_result = load_csv(sim_name_pf)
emt_result = load_csv(sim_name_emt)

print("Power-flow result:")
display(pf_result.head())

print("EMT result:")
display(emt_result.head())

# Plot active and reactive power

In [ ]:
time_column = emt_result.columns[0]
time = emt_result[time_column].to_numpy()

for prefix, ylabel, title in [
    ("P_elec", "P [W]", "VSI active power"),
    ("Q_elec", "Q [var]", "VSI reactive power"),
]:
    columns = signal_columns(emt_result, prefix)

    if not columns:
        print(f"Skipping {prefix}: no columns found.")
        continue

    plt.figure(figsize=(12, 5))

    for column in columns:
        plt.plot(time, emt_result[column], label=column)

    for event_time in [0.2, 0.4, 0.6, 0.8]:
        plt.axvline(event_time, linestyle="--")

    plt.xlabel("Time [s]")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

# Plot three-phase voltages

In [ ]:
voltage_signals = [
    ("PCC voltage", "Voltage_PCC"),
    ("Node 2 voltage", "Voltage_Node_2"),
    ("Node 3 voltage", "Voltage_Node_3"),
    ("Node 4 voltage", "Voltage_Node_4"),
    ("Internal source voltage", "Voltage_Source"),
]

for title, prefix in voltage_signals:
    columns = signal_columns(emt_result, prefix)

    if not columns:
        print(f"Skipping {title}: no columns found.")
        continue

    plt.figure(figsize=(12, 5))

    for column in columns:
        plt.plot(time, emt_result[column], label=column)

    for event_time in [0.2, 0.4, 0.6, 0.8]:
        plt.axvline(event_time, linestyle="--")

    plt.xlabel("Time [s]")
    plt.ylabel("Voltage [V]")
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

# Plot the three-phase voltage-vector magnitude

In [ ]:
for title, prefix in voltage_signals[:-1]:
    columns = signal_columns(emt_result, prefix)

    if len(columns) != 3:
        print(f"{title}: expected three phase columns, found {len(columns)}.")
        continue

    va = emt_result[columns[0]]
    vb = emt_result[columns[1]]
    vc = emt_result[columns[2]]

    voltage_magnitude = np.sqrt(2.0 / 3.0 * (va**2 + vb**2 + vc**2))

    plt.figure(figsize=(12, 5))
    plt.plot(time, voltage_magnitude, label="Voltage magnitude")
    plt.axhline(
        phase_voltage_peak,
        linestyle="--",
        label=f"Reference: {phase_voltage_peak:.2f} V",
    )

    for event_time in [0.2, 0.4, 0.6, 0.8]:
        plt.axvline(event_time, linestyle="--")

    plt.xlabel("Time [s]")
    plt.ylabel("Voltage magnitude [V]")
    plt.title(title)
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

# Plot interface current

In [ ]:
current_columns = signal_columns(emt_result, "Current_RLC")

if current_columns:
    plt.figure(figsize=(12, 5))

    for column in current_columns:
        plt.plot(time, emt_result[column], label=column)

    for event_time in [0.2, 0.4, 0.6, 0.8]:
        plt.axvline(event_time, linestyle="--")

    plt.xlabel("Time [s]")
    plt.ylabel("Current [A]")
    plt.title("VSI interface current")
    plt.grid(True)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()
else:
    print("No Current_RLC columns found.")